In [6]:
import sqlite3
import pandas as pd

DB_NAME = "cat_fleet_medallion.db"
conn = sqlite3.connect(DB_NAME)

# load gold table from yesterday
df = pd.read_sql("SELECT * FROM gold_telematics", conn)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['machine_id', 'timestamp']).reset_index(drop=True)

# ── 1. LAG FEATURES ────────────────────────────────────────────────────────
lag_cols = [
    'engine_temp_c', 'engine_rpm', 'oil_pressure_psi', 'coolant_temp_c',
    'vibration_level', 'battery_voltage_v', 'brake_temp_c', 'brake_pad_wear_mm'
]
for col in lag_cols:
    df[f'{col}_lag1'] = df.groupby('machine_id')[col].shift(1)

# ── 2. DELTA FEATURES ──────────────────────────────────────────────────────
for col in lag_cols:
    df[f'{col}_delta1'] = df[col] - df[f'{col}_lag1']

# ── 3. CUMULATIVE IDLING HOURS (capped to avoid overstating from data gaps) ──
df['time_diff_hours'] = df.groupby('machine_id')['timestamp'].diff().dt.total_seconds() / 3600

MAX_IDLING_GAP_HOURS = 1.0
df['capped_time_diff_hours'] = df['time_diff_hours'].clip(upper=MAX_IDLING_GAP_HOURS)
df['idling_hours_this_reading'] = df['is_idling'] * df['capped_time_diff_hours'].fillna(0)
df['cumulative_idling_hours'] = df.groupby('machine_id')['idling_hours_this_reading'].cumsum()

# ── 4. ADD MISSING ROLLING AVERAGES (brake + other sensors) ──────────────
extra_roll_cols = ['engine_temp_c', 'coolant_temp_c', 'vibration_level',
                    'brake_temp_c', 'brake_pad_wear_mm', 'brake_fluid_level_psi']
for col in extra_roll_cols:
    df[f'{col}_roll3_mean'] = (
        df.groupby('machine_id')[col]
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )

# ── 5. RE-SAVE ENRICHED GOLD TABLE ────────────────────────────────────────
df.to_sql("gold_telematics", conn, if_exists="replace", index=False)
print(f"Gold table enriched and saved: {len(df)} rows, {len(df.columns)} columns")

# ── AUDIT ──────────────────────────────────────────────────────────────────
sample = pd.read_sql(
    "SELECT machine_id, timestamp, engine_temp_c, engine_temp_c_lag1, "
    "engine_temp_c_delta1, time_diff_hours, cumulative_idling_hours FROM gold_telematics LIMIT 8",
    conn
)
print(sample.to_string(index=False))

conn.close()

Gold table enriched and saved: 1970 rows, 76 columns
machine_id           timestamp  engine_temp_c  engine_temp_c_lag1  engine_temp_c_delta1  time_diff_hours  cumulative_idling_hours
   VEH0000 2023-01-01 00:00:00      99.388213                 NaN                   NaN              NaN                      0.0
   VEH0000 2023-01-01 00:56:00      84.118706           99.388213            -15.269507         0.933333                      0.0
   VEH0000 2023-01-01 01:00:00      96.887263           84.118706             12.768558         0.066667                      0.0
   VEH0000 2023-01-01 01:12:00     105.465312           96.887263              8.578049         0.200000                      0.0
   VEH0000 2023-01-01 01:27:00      91.130527          105.465312            -14.334785         0.250000                      0.0
   VEH0000 2023-01-01 01:28:00     103.521431           91.130527             12.390904         0.016667                      0.0
   VEH0000 2023-01-01 02:06:00      9

In [7]:
# check if is_idling ever equals 1, and if cumulative_idling_hours ever increases
check = pd.read_sql(
    "SELECT machine_id, COUNT(*) as total_rows, SUM(is_idling) as idling_rows, "
    "MAX(cumulative_idling_hours) as max_cumulative_idling "
    "FROM gold_telematics GROUP BY machine_id LIMIT 10",
    sqlite3.connect('cat_fleet_medallion.db')
)
print(check)

  machine_id  total_rows  idling_rows  max_cumulative_idling
0    VEH0000          38            0               0.000000
1    VEH0001          44            0               0.000000
2    VEH0002          36            4               0.916667
3    VEH0003          37            1               0.700000
4    VEH0004          35            0               0.000000
5    VEH0005          39            1               0.500000
6    VEH0006          42            1               1.000000
7    VEH0007          38            2               1.366667
8    VEH0008          41            1               0.250000
9    VEH0009          38            1               0.083333


In [8]:
conn = sqlite3.connect('cat_fleet_medallion.db')
veh6 = pd.read_sql(
    "SELECT machine_id, timestamp, is_idling, time_diff_hours, idling_hours_this_reading "
    "FROM gold_telematics WHERE machine_id = 'VEH0006' ORDER BY timestamp",
    conn
)
print(veh6.to_string(index=False))
conn.close()

machine_id           timestamp  is_idling  time_diff_hours  idling_hours_this_reading
   VEH0006 2023-01-01 00:00:00          0              NaN                        0.0
   VEH0006 2023-01-01 00:37:00          0         0.616667                        0.0
   VEH0006 2023-01-01 00:48:00          0         0.183333                        0.0
   VEH0006 2023-01-01 00:56:00          0         0.133333                        0.0
   VEH0006 2023-01-01 01:35:00          0         0.650000                        0.0
   VEH0006 2023-01-01 01:44:00          0         0.150000                        0.0
   VEH0006 2023-01-01 01:44:00          0         0.000000                        0.0
   VEH0006 2023-01-01 02:13:00          0         0.483333                        0.0
   VEH0006 2023-01-01 02:48:00          0         0.583333                        0.0
   VEH0006 2023-01-01 02:50:00          0         0.033333                        0.0
   VEH0006 2023-01-01 02:56:00          0         0.10